# Roxy feature module tutorial on simulated data

This notebook demonstrates how to use the **Roxy feature module** on a
simulated classification dataset.

We will:

1. Generate a synthetic dataset with numeric and categorical features.
2. Infer feature types using `infer_feature_kinds`.
3. Apply scaling to numeric features using `ColumnScaler`.
4. Perform feature selection with different strategies using `FeatureSelector`.
5. Compare which features are retained by each strategy.


In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification

from roxy.features import (
    infer_feature_kinds,
    summarise_feature_kinds,
    ColumnScaler,
    FeatureSelector,
)

## 1. Generate a simulated classification dataset

We use `sklearn.datasets.make_classification` to generate a dataset with
numeric features, and then enrich it with a few categorical columns to
mimic a more realistic scenario (e.g. batches, groups, etc.).

In [2]:
# Generate a numeric classification dataset
X_num, y = make_classification(
    n_samples=300,
    n_features=20,
    n_informative=5,
    n_redundant=5,
    n_repeated=0,
    n_clusters_per_class=2,
    class_sep=1.5,
    flip_y=0.01,
    random_state=42,
)

num_cols = [f"f{i}" for i in range(1, X_num.shape[1] + 1)]
df_num = pd.DataFrame(X_num, columns=num_cols)

# Add some categorical columns
rng = np.random.default_rng(42)
groups = rng.choice(["A", "B", "C"], size=df_num.shape[0])
batches = rng.integers(1, 5, size=df_num.shape[0])

df = df_num.copy()
df["group"] = groups
df["batch"] = batches.astype(str)  # treat as categorical

target = pd.Series(y, name="label")

df.head()

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f13,f14,f15,f16,f17,f18,f19,f20,group,batch
0,-0.476221,-1.766291,4.683657,-0.582759,-2.896604,0.420925,-2.664844,0.390465,1.879017,0.874389,...,0.262561,0.193590,1.001412,-0.137372,-0.284809,0.265362,-1.017905,0.850898,A,2
1,-0.387131,6.328878,-0.088812,0.813205,2.480878,-3.663890,5.043361,-0.493123,3.422476,-0.045411,...,0.025388,-1.919673,3.519030,-0.689728,-5.392896,-1.256507,-6.390491,-0.013838,C,4
2,-0.256180,1.897775,-1.326633,-0.351921,1.408693,-0.805424,4.442243,-1.203201,0.429229,-0.241497,...,-0.061764,0.479442,0.354118,-0.649765,-4.218507,-0.487203,-1.314276,0.874517,B,1
3,-0.781156,-3.027477,1.190440,0.956703,-0.604138,2.172490,-0.794148,-1.151154,0.230931,-0.259800,...,1.598538,0.802128,-0.996606,-1.416227,-0.738354,-0.809381,1.789693,0.862224,B,4
4,0.441307,-2.358042,2.985085,0.148089,-0.969610,2.124255,-2.803224,0.638660,0.729964,-0.733156,...,0.228996,-1.857901,0.631577,0.298158,1.930282,0.367620,0.818977,0.603248,B,3


## 2. Infer feature types with `infer_feature_kinds`

We classify each column into semantic kinds (numeric, categorical, etc.)
to guide downstream scaling and feature selection steps.

In [3]:
kinds = infer_feature_kinds(df, max_categories=10)
summary_kinds = summarise_feature_kinds(kinds)
summary_kinds

,kind,dtype,n_unique,is_constant
name,,,,
batch,categorical,object,4,False
f1,numeric,float64,300,False
f10,numeric,float64,300,False
f11,numeric,float64,300,False
f12,numeric,float64,300,False
f13,numeric,float64,300,False
f14,numeric,float64,300,False
f15,numeric,float64,300,False
f16,numeric,float64,300,False


In [4]:
numeric_cols = [name for name, meta in kinds.items() if meta.kind.value == "numeric"]
categorical_cols = [name for name, meta in kinds.items() if meta.kind.value == "categorical"]

numeric_cols, categorical_cols

(['f1',
  'f2',
  'f3',
  'f4',
  'f5',
  'f6',
  'f7',
  'f8',
  'f9',
  'f10',
  'f11',
  'f12',
  'f13',
  'f14',
  'f15',
  'f16',
  'f17',
  'f18',
  'f19',
  'f20'],
 ['group', 'batch'])

## 3. Scale numeric features with `ColumnScaler`

We standardise only the numeric features, leaving categorical columns
untouched. The transformer preserves the DataFrame structure.

In [5]:
scaler = ColumnScaler(strategy="standard", columns=numeric_cols)
df_scaled = scaler.fit_transform(df)

df_scaled.head()

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f13,f14,f15,f16,f17,f18,f19,f20,group,batch
0,-0.412986,-0.180493,1.393683,-0.597739,-1.199337,-0.200193,-0.564615,0.391327,0.665294,0.888416,...,0.206720,0.133348,0.431898,-0.159006,0.155739,0.320293,-0.483261,0.888896,A,2
1,-0.321781,2.503624,-0.779657,0.761195,1.901536,-2.582649,1.526858,-0.464390,1.515512,-0.014716,...,-0.046462,-2.214814,1.634602,-0.697709,-2.090305,-1.145208,-2.786435,0.010445,C,4
2,-0.187722,1.034402,-1.343351,-0.373025,1.283271,-0.915457,1.363756,-1.152070,-0.133325,-0.207249,...,-0.139497,0.450973,0.122676,-0.658734,-1.573922,-0.404398,-0.610312,0.912890,B,1
3,-0.725160,-0.598665,-0.197097,0.900886,0.122591,0.821402,-0.057038,-1.101664,-0.242557,-0.225220,...,1.632876,0.809528,-0.522585,-1.406250,-0.043686,-0.714643,0.720328,0.900401,B,4
4,0.526322,-0.376700,0.620168,0.113722,-0.088154,0.793269,-0.602162,0.631693,0.032336,-0.689998,...,0.170890,-2.146176,0.255223,0.265758,1.129723,0.418763,0.304192,0.637318,B,3


## 4. Feature selection with `FeatureSelector`

We now apply different feature selection strategies on the scaled
numeric features, treating this as a classification task.

### 4.1 Variance threshold

In [6]:
selector_var = FeatureSelector(
    strategy="variance",
    task_type="classification",
    columns=numeric_cols,
    selector_kwargs={"threshold": 0.0},  # remove only truly constant features
)
X_var = selector_var.fit_transform(df_scaled, target)

selector_var.selected_columns_, X_var.shape

(['f1',
  'f2',
  'f3',
  'f4',
  'f5',
  'f6',
  'f7',
  'f8',
  'f9',
  'f10',
  'f11',
  'f12',
  'f13',
  'f14',
  'f15',
  'f16',
  'f17',
  'f18',
  'f19',
  'f20'],
 (300, 20))

### 4.2 K-best (ANOVA F-score)

We select the top 10 numeric features according to the ANOVA F-score
with respect to the class labels.

In [7]:
selector_kbest = FeatureSelector(
    strategy="kbest",
    task_type="classification",
    columns=numeric_cols,
    selector_kwargs={"k": 10, "score": "f"},
)
X_kbest = selector_kbest.fit_transform(df_scaled, target)

selector_kbest.selected_columns_, X_kbest.shape

(['f2', 'f3', 'f5', 'f6', 'f7', 'f9', 'f11', 'f17', 'f19', 'f20'], (300, 10))

### 4.3 K-best (mutual information)

We can also use mutual information instead of ANOVA to capture possible
non-linear relationships between features and the target.

In [8]:
selector_mi = FeatureSelector(
    strategy="mutual_info",
    task_type="classification",
    columns=numeric_cols,
    selector_kwargs={"k": 10},
)
X_mi = selector_mi.fit_transform(df_scaled, target)

selector_mi.selected_columns_, X_mi.shape

(['f2', 'f3', 'f5', 'f6', 'f7', 'f9', 'f11', 'f15', 'f17', 'f19'], (300, 10))

### 4.4 Model-based selection (tree ensemble)

Finally, we use a tree-based ensemble (RandomForest) wrapped in
`SelectFromModel` to select features according to their importance
weights.

In [9]:
selector_tree = FeatureSelector(
    strategy="model_tree",
    task_type="classification",
    columns=numeric_cols,
    selector_kwargs={"threshold": "median"},  # keep features above median importance
)
X_tree = selector_tree.fit_transform(df_scaled, target)

selector_tree.selected_columns_, X_tree.shape

(['f2', 'f3', 'f5', 'f6', 'f7', 'f9', 'f11', 'f15', 'f17', 'f19'], (300, 10))

## 5. Comparing selected features across strategies

We can now compare which features were selected by each strategy. This
helps understand how sensitive the selection is to the underlying
criterion (variance, ANOVA, mutual information, model-based).

In [10]:
selected_sets = {
    "variance": set(selector_var.selected_columns_),
    "kbest_f": set(selector_kbest.selected_columns_),
    "kbest_mi": set(selector_mi.selected_columns_),
    "model_tree": set(selector_tree.selected_columns_),
}

selected_sets

{'variance': {'f1',
  'f10',
  'f11',
  'f12',
  'f13',
  'f14',
  'f15',
  'f16',
  'f17',
  'f18',
  'f19',
  'f2',
  'f20',
  'f3',
  'f4',
  'f5',
  'f6',
  'f7',
  'f8',
  'f9'},
 'kbest_f': {'f11', 'f17', 'f19', 'f2', 'f20', 'f3', 'f5', 'f6', 'f7', 'f9'},
 'kbest_mi': {'f11', 'f15', 'f17', 'f19', 'f2', 'f3', 'f5', 'f6', 'f7', 'f9'},
 'model_tree': {'f11',
  'f15',
  'f17',
  'f19',
  'f2',
  'f3',
  'f5',
  'f6',
  'f7',
  'f9'}}

In [11]:
# Features selected by all strategies
common_features = set.intersection(*selected_sets.values())
common_features

{'f11', 'f17', 'f19', 'f2', 'f3', 'f5', 'f6', 'f7', 'f9'}

In [12]:
# Features unique to each strategy (relative to union of all)
all_selected = set.union(*selected_sets.values())
unique_per_strategy = {
    name: sorted(list(feats - (all_selected - feats)))
    for name, feats in selected_sets.items()
}
unique_per_strategy

{'variance': ['f1',
  'f10',
  'f11',
  'f12',
  'f13',
  'f14',
  'f15',
  'f16',
  'f17',
  'f18',
  'f19',
  'f2',
  'f20',
  'f3',
  'f4',
  'f5',
  'f6',
  'f7',
  'f8',
  'f9'],
 'kbest_f': ['f11', 'f17', 'f19', 'f2', 'f20', 'f3', 'f5', 'f6', 'f7', 'f9'],
 'kbest_mi': ['f11', 'f15', 'f17', 'f19', 'f2', 'f3', 'f5', 'f6', 'f7', 'f9'],
 'model_tree': ['f11',
  'f15',
  'f17',
  'f19',
  'f2',
  'f3',
  'f5',
  'f6',
  'f7',
  'f9']}